# CLIP + AST + ontology — independent evaluation 3500

Frozen basic-ontology evaluation using the same modular structure as the development notebook. The ontology is loaded externally from `../Acoustic_ontology.ipynb`.

The evaluation keeps the uploaded notebook's fixed decision settings:
- ontology normalization: `max`
- CLIP protection gap: `candidate`
- event evidence: `confidence × persistence × semantic strength`
- ontology evidence is restricted to the CLIP Top-3 candidate set


## Setup and frozen parameters

In [9]:
%run ../Acoustic_ontology.ipynb


Ontology rules: 22
Ontology event mappings: 22


In [10]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

# Project paths
PROJECT_ROOT = Path(r"C:\Users\22313906t\Desktop\TAU_project")
CLIP_PATH = PROJECT_ROOT / 'results_CLIP_TEST_3500' / 'tau_3500_test_clip_predictions.csv'
AST_PATH = PROJECT_ROOT / 'results_AST_TEST_3500' / 'tau_3500_test_ast_events.csv'
OUTPUT_DIR = PROJECT_ROOT / 'results_ONTOLOGY_TEST_3500'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / 'tau_3500_test_clip_ast_ontology.csv'
SUMMARY_PATH = OUTPUT_DIR / 'tau_3500_test_clip_ast_ontology_summary.csv'
PER_CLASS_PATH = OUTPUT_DIR / 'tau_3500_test_clip_ast_ontology_per_class.csv'
CONFIG_PATH = OUTPUT_DIR / 'tau_3500_test_clip_ast_ontology_config.json'

# Dataset settings
TAU_CLASSES = [
    'airport',
    'bus',
    'metro',
    'metro station',
    'park',
    'public square',
    'shopping mall',
    'pedestrian street',
    'street traffic',
    'tram',
]
EXPECTED_TOTAL = 3500
N_PER_CLASS = 350
CLIP_SCORE_COLUMNS = {scene: scene.replace(' ', '_') + '_score' for scene in TAU_CLASSES}

# Basic-ontology parameters
MIN_EVENT_CONFIDENCE = 0.08
ONTOLOGY_LAMBDA = 0.4
CLIP_PROTECTION_MARGIN = 0.2
MIN_ONTOLOGY_EVIDENCE = 0.15
MIN_ONTOLOGY_ADVANTAGE = 0.05
STRONG_ONTOLOGY_EVIDENCE = 0.6

# Selectable modes
# Ontology score normalization
NORMALIZATION_MODE = 'max'

# Candidate-specific CLIP protection
CLIP_PROTECTION_MODE = 'candidate'
print('=' * 78)
print('TAU 3500 TEST — CLIP + AST + FROZEN BASIC ONTOLOGY')
print('=' * 78)
print('\nCLIP:')
print(CLIP_PATH)
print('\nAST:')
print(AST_PATH)
print('\nOutput:')
print(OUTPUT_DIR)
print('\nOntology normalization:', NORMALIZATION_MODE.upper())
print('CLIP protection mode:', CLIP_PROTECTION_MODE)


TAU 3500 TEST — CLIP + AST + FROZEN BASIC ONTOLOGY

CLIP:
C:\Users\22313906t\Desktop\TAU_project\results_CLIP_TEST_3500\tau_3500_test_clip_predictions.csv

AST:
C:\Users\22313906t\Desktop\TAU_project\results_AST_TEST_3500\tau_3500_test_ast_events.csv

Output:
C:\Users\22313906t\Desktop\TAU_project\results_ONTOLOGY_TEST_3500

Ontology normalization: MAX
CLIP protection mode: candidate


## Load and prepare CLIP/AST results

In [11]:
# Load CLIP results
if not CLIP_PATH.exists():
    raise FileNotFoundError(f'Cannot find CLIP predictions:\n{CLIP_PATH}')
if not AST_PATH.exists():
    raise FileNotFoundError(f'Cannot find AST events:\n{AST_PATH}')
clip_df = pd.read_csv(CLIP_PATH)
required_clip_columns = {'clip_id', 'ground_truth', 'clip_top1', 'clip_top2', 'clip_top3'}
missing_clip_columns = required_clip_columns - set(clip_df.columns)
if missing_clip_columns:
    raise ValueError(f'Missing CLIP columns: {missing_clip_columns}')
clip_df['clip_id'] = clip_df['clip_id'].astype(str)
clip_df['ground_truth'] = clip_df['ground_truth'].astype(str)
clip_df = clip_df.drop_duplicates(subset=['clip_id'], keep='last').reset_index(drop=True)
for scene, column in CLIP_SCORE_COLUMNS.items():
    if column not in clip_df.columns:
        raise ValueError(f'Missing CLIP score column: {column}')

# Validate the dataset
print('\n' + '=' * 78)
print('CLIP DATASET CHECK')
print('=' * 78)
print('CLIP rows:', len(clip_df))
print('Unique IDs:', clip_df['clip_id'].nunique())
if len(clip_df) != EXPECTED_TOTAL:
    raise RuntimeError(f'Expected {EXPECTED_TOTAL} CLIP rows, found {len(clip_df)}.')
class_counts = clip_df['ground_truth'].value_counts().reindex(TAU_CLASSES, fill_value=0)
print('\nClass distribution:')
print(class_counts)
if not (class_counts == N_PER_CLASS).all():
    raise RuntimeError(f'Expected {N_PER_CLASS} clips per class.')

# Load AST results
ast_df = pd.read_csv(AST_PATH)
if 'clip_id' not in ast_df.columns:
    raise ValueError('AST file has no clip_id column.')
ast_df['clip_id'] = ast_df['clip_id'].astype(str)

# Detect AST event and score columns
EVENT_COL = None
for candidate in ['label', 'event', 'event_label', 'audio_event', 'class_name']:
    if candidate in ast_df.columns:
        EVENT_COL = candidate
        break
if EVENT_COL is None:
    raise ValueError(f'Cannot find AST event column. Available columns: {list(ast_df.columns)}')
SCORE_COL = None
for candidate in ['max_score', 'score', 'mean_score']:
    if candidate in ast_df.columns:
        SCORE_COL = candidate
        break
if SCORE_COL is None:
    raise ValueError(f'Cannot find AST score column. Available columns: {list(ast_df.columns)}')
print('\nAST event column:', EVENT_COL)
print('AST score column:', SCORE_COL)

# Persistence handling
# old behavior:
# use an existing persistence column if present;
# otherwise compute count / num_chunks;
# if neither is available, stop instead of silently using 1.0.
if 'persistence' not in ast_df.columns:
    if 'count' in ast_df.columns and 'num_chunks' in ast_df.columns:
        ast_df[
            'persistence'] = ast_df['count'].astype(float) / ast_df['num_chunks'].replace(0,
            np.nan).astype(float,
        )
        ast_df['persistence'] = ast_df['persistence'].fillna(0.0)
    else:
        raise ValueError('AST file needs either persistence or count + num_chunks.')

# Filter AST to CLIP ids
clip_ids = set(clip_df['clip_id'])
ast_df = ast_df[ast_df['clip_id'].isin(clip_ids)].copy()
ast_matched_ids = set(ast_df['clip_id'])
missing_ast_ids = clip_ids - ast_matched_ids
print('\n' + '=' * 78)
print('AST / CLIP MATCH CHECK')
print('=' * 78)
print('CLIP clips:', len(clip_ids))
print('AST matched clips:', len(ast_matched_ids))
print('Missing AST clips:', len(missing_ast_ids))
if missing_ast_ids:
    print('\nFirst missing AST IDs:')
    for clip_id in sorted(missing_ast_ids)[:20]:
        print(clip_id)
    raise RuntimeError('AST results do not cover all CLIP clips.')

# Normalize AST event names and numeric fields
ast_df['event_norm'] = ast_df[EVENT_COL].apply(normalize_event)
ast_df['event_score'] = pd.to_numeric(ast_df[SCORE_COL], errors='coerce').fillna(0.0)
ast_df['persistence'] = pd.to_numeric(ast_df['persistence'], errors='coerce').fillna(0.0)

# Index AST by CLIP
ast_groups = {clip_id: group.copy() for clip_id, group in ast_df.groupby('clip_id')}



CLIP DATASET CHECK
CLIP rows: 3500
Unique IDs: 3500

Class distribution:
ground_truth
airport              350
bus                  350
metro                350
metro station        350
park                 350
public square        350
shopping mall        350
pedestrian street    350
street traffic       350
tram                 350
Name: count, dtype: int64

AST event column: label
AST score column: max_score

AST / CLIP MATCH CHECK
CLIP clips: 3500
AST matched clips: 3500
Missing AST clips: 0


## Helper functions

In [12]:
# Safe float
def safe_float(value, default=0.0):
    try:
        value = float(value)
        if np.isfinite(value):
            return value
    except Exception:
        pass
    return default

# Ontology normalization
# "sum" = historical old-code behavior
# "max" = optional comparison
def normalize_ontology_scores(ontology_scores, candidates, mode='sum'):
    if mode == 'sum':
        denominator = sum((ontology_scores[scene] for scene in candidates))
    elif mode == 'max':
        denominator = max((ontology_scores[scene] for scene in candidates))
    else:
        raise ValueError("normalization mode must be 'sum' or 'max'")
    if denominator > 0:
        return {scene: ontology_scores[scene] / denominator for scene in candidates}
    return {scene: 0.0 for scene in candidates}

# CLIP protection gap
# "candidate" = historical old-code behavior:
# Top-1 CLIP score - ontology-proposed candidate CLIP score
# "top2" = optional newer comparison:
# Top-1 CLIP score - Top-2 CLIP score
def compute_clip_protection_gap(clip_scores, candidates, proposed_winner, mode='candidate'):
    original_top1 = candidates[0]
    original_score = clip_scores[original_top1]
    if mode == 'candidate':
        comparison_scene = proposed_winner
    elif mode == 'top2':
        if len(candidates) >= 2:
            comparison_scene = candidates[1]
        else:
            comparison_scene = original_top1
    else:
        raise ValueError("CLIP protection mode must be 'candidate' or 'top2'")
    comparison_score = clip_scores[comparison_scene]
    visual_gap = original_score - comparison_score
    return max(0.0, visual_gap)


## Basic ontology reasoning

In [13]:
# Run basic ontology for one CLIP
def run_ontology_for_clip(
    clip_row,
    audio_group,
    normalization_mode='sum',
    clip_protection_mode='candidate',
):
    candidates = [str(clip_row['clip_top1']), str(clip_row['clip_top2']), str(clip_row['clip_top3'])]
    candidates = list(dict.fromkeys(candidates))
    original_top1 = candidates[0]
    clip_scores = {scene: safe_float(clip_row[CLIP_SCORE_COLUMNS[scene]]) for scene in candidates}
    ontology_scores = {scene: 0.0 for scene in candidates}
    support_scores = {scene: 0.0 for scene in candidates}
    refine_scores = {scene: 0.0 for scene in candidates}
    fired_rules = []
    ignored_events = []
    evidence_rows = []
    if audio_group is not None:
        for _, event_row in audio_group.iterrows():
            event = event_row['event_norm']
            confidence = safe_float(event_row['event_score'])
            persistence = safe_float(event_row['persistence'])
            if confidence < MIN_EVENT_CONFIDENCE:
                continue
            rule = RULE_LOOKUP.get(event)
            if rule is None:
                continue
            if rule['relation'] == 'IGNORE':
                ignored_events.append(f'{event}({confidence:.3f})')
                continue
            relation = rule['relation']
            for target_scene, semantic_strength in rule['targets'].items():
                if target_scene not in candidates:
                    continue
                evidence = confidence * persistence * semantic_strength
                ontology_scores[target_scene] += evidence
                if relation == 'SUPPORT':
                    support_scores[target_scene] += evidence
                elif relation == 'REFINE':
                    refine_scores[target_scene] += evidence
                fired_rules.append(f"{rule['name']}:{event}->{target_scene}[{relation};conf={confidence:.3f};pers={persistence:.2f};sem={semantic_strength:.2f};E={evidence:.3f}]")
                evidence_rows.append({'clip_id': str(clip_row['clip_id']), 'event': event_row[EVENT_COL], 'event_norm': event, 'event_max_score': confidence, 'persistence': persistence, 'ontology_rule': rule['name'], 'relation': relation, 'target_scene': target_scene, 'semantic_strength': semantic_strength, 'evidence': evidence})
    ontology_normalized = normalize_ontology_scores(
        ontology_scores=ontology_scores,
        candidates=candidates,
        mode=normalization_mode,
    )
    adjusted_scores = {scene: clip_scores[scene] + ONTOLOGY_LAMBDA * ontology_normalized[scene] for scene in candidates}
    proposed_winner = max(candidates, key=lambda scene: adjusted_scores[scene])
    clip_protection_gap = compute_clip_protection_gap(
        clip_scores=clip_scores,
        candidates=candidates,
        proposed_winner=proposed_winner,
        mode=clip_protection_mode,
    )
    current_ontology = ontology_scores[original_top1]
    alternative_ontology = ontology_scores[proposed_winner]
    ontology_advantage = alternative_ontology - current_ontology
    final_prediction = original_top1
    decision = 'KEEP_CLIP'
    reason = 'CLIP Top-1 retained'
    if proposed_winner != original_top1:
        enough_ontology = alternative_ontology >= MIN_ONTOLOGY_EVIDENCE
        enough_advantage = ontology_advantage >= MIN_ONTOLOGY_ADVANTAGE
        visual_gap_allowed = clip_protection_gap <= CLIP_PROTECTION_MARGIN
        strong_audio = alternative_ontology >= STRONG_ONTOLOGY_EVIDENCE
        if enough_ontology and enough_advantage and (visual_gap_allowed or strong_audio):
            final_prediction = proposed_winner
            decision = 'ONTOLOGY_RERANK'
            if strong_audio:
                reason = 'Strong ontology evidence overrode CLIP protection margin'
            else:
                reason = 'Ontology evidence corrected uncertain CLIP Top-1'
        elif not enough_ontology:
            reason = 'Alternative ontology evidence below minimum'
        elif not enough_advantage:
            reason = 'Ontology advantage below minimum'
        elif not (visual_gap_allowed or strong_audio):
            reason = 'CLIP protection margin blocked correction'
    output = {
        'ontology_prediction': final_prediction,
        'ontology_decision': decision,
        'ontology_reason': reason,
        'ontology_normalization': normalization_mode,
        'clip_protection_mode': clip_protection_mode,
        'candidate_after_fusion': proposed_winner,
        'clip_margin_to_candidate': clip_protection_gap,
        'ontology_advantage': ontology_advantage,
        'current_ontology_evidence': current_ontology,
        'alternative_ontology_evidence': alternative_ontology,
        'fired_rule_count': len(fired_rules),
        'fired_rules': ' | '.join(fired_rules),
        'ignored_events': ' | '.join(ignored_events),
        '_evidence_rows': evidence_rows,
    }
    for candidate_index in range(3):
        n = candidate_index + 1
        if candidate_index < len(candidates):
            scene = candidates[candidate_index]
            output[f'candidate{n}'] = scene
            output[f'candidate{n}_clip_score'] = clip_scores[scene]
            output[f'candidate{n}_ontology_raw'] = ontology_scores[scene]
            output[f'candidate{n}_ontology_norm'] = ontology_normalized[scene]
            output[f'candidate{n}_support'] = support_scores[scene]
            output[f'candidate{n}_refine'] = refine_scores[scene]
            output[f'candidate{n}_adjusted'] = adjusted_scores[scene]
        else:
            output[f'candidate{n}'] = np.nan
            output[f'candidate{n}_clip_score'] = np.nan
            output[f'candidate{n}_ontology_raw'] = np.nan
            output[f'candidate{n}_ontology_norm'] = np.nan
            output[f'candidate{n}_support'] = np.nan
            output[f'candidate{n}_refine'] = np.nan
            output[f'candidate{n}_adjusted'] = np.nan
    return output


## Run, evaluate, and save

In [14]:
# Run all clips
result_rows = []
all_evidence_rows = []
for idx, clip_row in clip_df.iterrows():
    clip_id = str(clip_row['clip_id'])
    audio_group = ast_groups.get(clip_id)
    ontology_result = run_ontology_for_clip(
        clip_row=clip_row,
        audio_group=audio_group,
        normalization_mode=NORMALIZATION_MODE,
        clip_protection_mode=CLIP_PROTECTION_MODE,
    )
    evidence_rows = ontology_result.pop('_evidence_rows')
    all_evidence_rows.extend(evidence_rows)
    result_rows.append({'clip_id': clip_id, 'ground_truth': clip_row['ground_truth'], 'clip_top1': clip_row['clip_top1'], 'clip_top1_score': clip_row['clip_top1_score'], 'clip_top2': clip_row['clip_top2'], 'clip_top2_score': clip_row['clip_top2_score'], 'clip_top3': clip_row['clip_top3'], 'clip_top3_score': clip_row['clip_top3_score'], **ontology_result})
    if (idx + 1) % 500 == 0:
        print(f'Processed {idx + 1}/{len(clip_df)}')
result_df = pd.DataFrame(result_rows)
evidence_df = pd.DataFrame(all_evidence_rows)

# Correctness and change types
result_df['clip_correct'] = result_df['clip_top1'] == result_df['ground_truth']
result_df['ontology_correct'] = result_df['ontology_prediction'] == result_df['ground_truth']
result_df['changed'] = result_df['clip_top1'] != result_df['ontology_prediction']
result_df[
    'change_type'] = np.select([result_df['changed'] & ~result_df['clip_correct'] & result_df['ontology_correct'], result_df['changed'] & result_df['clip_correct'] & ~result_df['ontology_correct'], result_df['changed'] & ~result_df['clip_correct'] & ~result_df['ontology_correct'], result_df['changed'] & result_df['clip_correct'] & result_df['ontology_correct']],
    ['Wrong->Correct', 'Correct->Wrong', 'Wrong->Wrong', 'Correct->Correct'],
    default='UNCHANGED',
)

# Metrics
clip_accuracy = accuracy_score(result_df['ground_truth'], result_df['clip_top1'])
ontology_accuracy = accuracy_score(result_df['ground_truth'], result_df['ontology_prediction'])
clip_f1 = f1_score(
    result_df['ground_truth'],
    result_df['clip_top1'],
    labels=TAU_CLASSES,
    average='macro',
    zero_division=0,
)
ontology_f1 = f1_score(
    result_df['ground_truth'],
    result_df['ontology_prediction'],
    labels=TAU_CLASSES,
    average='macro',
    zero_division=0,
)

# Correction analysis
change_counts = result_df['change_type'].value_counts()
wrong_correct = int(change_counts.get('Wrong->Correct', 0))
correct_wrong = int(change_counts.get('Correct->Wrong', 0))
wrong_wrong = int(change_counts.get('Wrong->Wrong', 0))
correct_correct = int(change_counts.get('Correct->Correct', 0))
number_changed = int(result_df['changed'].sum())
net_gain = wrong_correct - correct_wrong

# Proper correction precision
correction_attempts = wrong_correct + correct_wrong
correction_precision = wrong_correct / correction_attempts if correction_attempts > 0 else 0.0

# old notebook called this quantity
# "correction precision"; it is more accurately
# the beneficial-change rate.
beneficial_change_rate = wrong_correct / number_changed if number_changed > 0 else 0.0

# Per-class results
per_class_rows = []
for scene in TAU_CLASSES:
    temp = result_df[result_df['ground_truth'] == scene].copy()
    clip_acc = (temp['clip_top1'] == scene).mean()
    ontology_acc = (temp['ontology_prediction'] == scene).mean()
    scene_changed = temp[temp['changed']]
    scene_wc = int((scene_changed['change_type'] == 'Wrong->Correct').sum())
    scene_cw = int((scene_changed['change_type'] == 'Correct->Wrong').sum())
    scene_ww = int((scene_changed['change_type'] == 'Wrong->Wrong').sum())
    per_class_rows.append({'class': scene, 'number_of_clips': len(temp), 'clip_accuracy_percent': clip_acc * 100, 'ontology_accuracy_percent': ontology_acc * 100, 'change_points': (ontology_acc - clip_acc) * 100, 'changed_predictions': len(scene_changed), 'wrong_to_correct': scene_wc, 'correct_to_wrong': scene_cw, 'wrong_to_wrong': scene_ww, 'net_gain': scene_wc - scene_cw})
per_class_df = pd.DataFrame(per_class_rows)

# Save outputs
result_df.to_csv(OUTPUT_PATH, index=False)
per_class_df.to_csv(PER_CLASS_PATH, index=False)
summary_df = pd.DataFrame([{'experiment': 'independent_test_3500', 'method': 'CLIP+AST+Ontology', 'number_of_clips': len(result_df), 'clip_accuracy': clip_accuracy, 'clip_accuracy_percent': clip_accuracy * 100, 'ontology_accuracy': ontology_accuracy, 'ontology_accuracy_percent': ontology_accuracy * 100, 'improvement_vs_clip_points': (ontology_accuracy - clip_accuracy) * 100, 'clip_macro_f1': clip_f1, 'ontology_macro_f1': ontology_f1, 'changed_predictions': number_changed, 'wrong_to_correct': wrong_correct, 'correct_to_wrong': correct_wrong, 'wrong_to_wrong': wrong_wrong, 'correct_to_correct': correct_correct, 'net_gain': net_gain, 'correction_precision': correction_precision, 'beneficial_change_rate': beneficial_change_rate, 'ontology_reranks': int((result_df['ontology_decision'] == 'ONTOLOGY_RERANK').sum()), 'ontology_normalization': NORMALIZATION_MODE, 'clip_protection_mode': CLIP_PROTECTION_MODE}])
summary_df.to_csv(SUMMARY_PATH, index=False)
config = {
    'experiment': 'independent_test_3500',
    'expected_total': EXPECTED_TOTAL,
    'clips_per_class': N_PER_CLASS,
    'audio_model': 'AST',
    'clip_path': str(CLIP_PATH),
    'ast_path': str(AST_PATH),
    'min_event_confidence': MIN_EVENT_CONFIDENCE,
    'ontology_lambda': ONTOLOGY_LAMBDA,
    'clip_protection_margin': CLIP_PROTECTION_MARGIN,
    'min_ontology_evidence': MIN_ONTOLOGY_EVIDENCE,
    'min_ontology_advantage': MIN_ONTOLOGY_ADVANTAGE,
    'strong_ontology_evidence': STRONG_ONTOLOGY_EVIDENCE,
    'candidate_policy': 'CLIP Top-3 only',
    'evidence_formula': 'confidence * persistence * semantic_strength',
    'ontology_normalization': NORMALIZATION_MODE,
    'clip_protection_mode': CLIP_PROTECTION_MODE,
}
with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

# Print results
print('\n' + '=' * 78)
print('CLIP + AST + ONTOLOGY RESULT')
print('=' * 78)
print(f'Ontology normalization:          {NORMALIZATION_MODE.upper()}')
print(f'CLIP protection mode:            {CLIP_PROTECTION_MODE}')
print(f'Number of clips:                 {len(result_df)}')
print(f'\nCLIP accuracy:                   {clip_accuracy * 100:.4f}%')
print(f'Ontology accuracy:               {ontology_accuracy * 100:.4f}%')
print(f'Improvement vs CLIP:             {(ontology_accuracy - clip_accuracy) * 100:+.4f} points')
print(f'\nCLIP Macro-F1:                  {clip_f1:.6f}')
print(f'Ontology Macro-F1:              {ontology_f1:.6f}')
print('\nCorrection analysis:')
print('Predictions changed:            ', number_changed)
print('Wrong -> Correct:               ', wrong_correct)
print('Correct -> Wrong:               ', correct_wrong)
print('Wrong -> Wrong:                 ', wrong_wrong)
print('Correct -> Correct change:      ', correct_correct)
print('Net correction gain:            ', f'{net_gain:+d}')
print('Correction precision:           ', f'{correction_precision * 100:.2f}%')
print('Beneficial-change rate:         ', f'{beneficial_change_rate * 100:.2f}%')
print('\nPer-class:')
print(per_class_df.round(3).to_string(index=False))
print('\nSaved:')
print(OUTPUT_PATH)
print(SUMMARY_PATH)
print(PER_CLASS_PATH)
print(CONFIG_PATH)


Processed 500/3500
Processed 1000/3500
Processed 1500/3500
Processed 2000/3500
Processed 2500/3500
Processed 3000/3500
Processed 3500/3500

CLIP + AST + ONTOLOGY RESULT
Ontology normalization:          MAX
CLIP protection mode:            candidate
Number of clips:                 3500

CLIP accuracy:                   77.0286%
Ontology accuracy:               77.8000%
Improvement vs CLIP:             +0.7714 points

CLIP Macro-F1:                  0.767533
Ontology Macro-F1:              0.775164

Correction analysis:
Predictions changed:             43
Wrong -> Correct:                33
Correct -> Wrong:                6
Wrong -> Wrong:                  4
Correct -> Correct change:       0
Net correction gain:             +27
Correction precision:            84.62%
Beneficial-change rate:          76.74%

Per-class:
            class  number_of_clips  clip_accuracy_percent  ontology_accuracy_percent  change_points  changed_predictions  wrong_to_correct  correct_to_wrong  wrong_to_wr